# 무역 데이터 유효성 검증 (Trade Data Validation Test)

미국 수출입 HS코드 데이터가 **기업 매출을 설명/예측하는 데 실제로 쓸모가 있는지** 단계별로
검증합니다. 상관계수 스캔 결과를 그대로 믿지 않고, 각 단계마다 "잡음으로 설명되는가"를
직접 확인합니다.

이 노트북은 SELECT만 사용하며 DB에 아무것도 쓰지 않습니다.

## 검증 단계

| § | 내용 | 통과 기준 |
|---|---|---|
| 1 | 데이터 무결성 | 회계 캘린더 스냅 / 중복 / 수출 self-join / 개정 이력 |
| 2 | 패널 구축 | 매출 YoY 3그룹 + 무역 YoY 패널 |
| 3 | **잡음 바닥** | 관측 상관계수 분포가 귀무분포를 초과하는가 |
| 4 | **사전 지정 가설** | 산업 논리로 미리 정한 쌍이 상위에 오는가 |
| 5 | 선행/동행 구조 | 최고점 위치와 그 식별 가능성 |
| 6 | **시점 정합 백테스트** | AR(1) 대비 증분 정보가 있는가 (Clark-West) |
| 7 | 방향성·전환점 | 전환점 구간에서 더 크게 기여하는가 |
| 8 | 괴리 탐지 | 무역-매출이 벌어지는 구간 포착 |
| 9 | 종합 스코어카드 | 단계별 판정 요약 |

## 핵심 원칙

1. **명목 표본 수를 믿지 않는다.** YoY 계열은 자기상관이 0.6~0.85라 n=49의 유효 표본은
   15~20 수준이다. 일반 t검정 p값은 수십~수백 배 과대평가된다.
2. **사후에 기준을 낮추지 않는다.** 각 §마다 통과 기준을 먼저 출력하고 판정한다.
3. **벤치마크는 0이 아니다.** 매출 YoY는 지속성이 높아 AR(1)만으로도 강력하다.
   무역 데이터의 가치는 AR(1) **위에** 얹는 증분 정보로만 측정한다.
4. **가설은 데이터를 보기 전에 정한다.** §4의 HYPOTHESES는 산업 논리로 미리 작성한다.

## 실행 순서
§1 → §2 를 먼저 돌린 뒤, §3 이후는 독립적으로 실행 가능합니다.


## §0. 설정

In [ ]:

# -*- coding: utf-8 -*-
from __future__ import annotations
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")


def _find_project_root(module_name="DATA", max_up=6):
    here = Path.cwd()
    for base in [here, *list(here.parents)[:max_up]]:
        if (base / module_name).is_dir():
            return base
    return None


try:
    from DATA.stock_invest_function import *
except ModuleNotFoundError:
    _root = _find_project_root("DATA")
    if _root is None:
        raise ModuleNotFoundError("DATA 패키지를 찾을 수 없습니다.")
    sys.path.insert(0, str(_root))
    from DATA.stock_invest_function import *
    print(f"[경로 자동 보정] {_root}")

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

try:
    from scipy import stats as sps
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("[경고] scipy 미설치 — 일부 검정을 건너뜁니다. pip install scipy")

db_info = {"host": get_db_host(), "port": 3307, "user": "stox7412",
           "password": "Apt106503!~", "database": "investar"}
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
    f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4",
    pool_pre_ping=True, pool_recycle=1800,
    connect_args={"connect_timeout": 10, "read_timeout": 120, "write_timeout": 60})

TABLE_REVENUE        = "US_IS_from_FMP"
TABLE_EXPORT_MONTHLY = "us_trade_export_monthly_with_forecast"
TABLE_IMPORT_MONTHLY = "us_trade_import_monthly_with_forecast"

# ---------------- 검증 설정 ----------------
DIRECTION         = "export"   # "export" 또는 "import"
MIN_QUARTERS      = 20         # 매출 연속 분기 최소
QUARTER_TOL_DAYS  = 100
MIN_MONTHS        = 60         # 무역 연속 개월 최소
MONTH_TOL_DAYS    = 35
MIN_CORR_PERIODS  = 12         # 상관계수 계산 최소 관측 수
N_SIM             = 100_000    # 귀무분포 시뮬레이션 횟수
RNG               = np.random.default_rng(20260727)

print(f"검증 대상: {DIRECTION.upper()}")


In [ ]:

# ============================================================
# 공통 헬퍼
# ============================================================

def _month_end_calendar(ts):
    """달력 월말. **무역 월별 데이터 전용** (월초 저장분을 해당 월말로)."""
    return pd.Timestamp(ts) + pd.offsets.MonthEnd(0)


def _month_end_fiscal(ts):
    """가장 가까운 월말. **기업 분기말 전용** (52/53주 회계 캘린더 대응).
        2023-07-01 -> 2023-06-30 | 2023-09-30 -> 2023-09-30 | 2021-05-02 -> 2021-04-30
    """
    return (pd.Timestamp(ts) - pd.Timedelta(days=15)) + pd.offsets.MonthEnd(0)


def get_trailing_consecutive_run(dates, tol_days=100):
    dates = sorted(pd.to_datetime(d) for d in dates)
    if not dates:
        return []
    run = [dates[-1]]
    for i in range(len(dates) - 2, -1, -1):
        if (run[-1] - dates[i]).days <= tol_days:
            run.append(dates[i])
        else:
            break
    return sorted(run)


def compute_revenue_yoy(series, tol_days=45):
    series = series.sort_index()
    idx = series.index
    base = series.reindex(idx - pd.DateOffset(years=1), method="nearest",
                          tolerance=pd.Timedelta(days=tol_days))
    base.index = idx
    with np.errstate(invalid="ignore", divide="ignore"):
        yoy = (series.values / base.values - 1.0) * 100.0
    return pd.Series(yoy, index=idx).replace([np.inf, -np.inf], np.nan).dropna()


def _export_version_col(conn):
    cols = [r[0] for r in conn.execute(text(f"SHOW COLUMNS FROM {TABLE_EXPORT_MONTHLY}")).fetchall()]
    return "created_at" if "created_at" in cols else ("input_date" if "input_date" in cols else None)


def load_trade_monthly_long(engine, direction="export", verbose=True):
    """무역 월별 로드. 수출 self-join은 (hs_code, date_month_end) 단위로 최신본만."""
    if direction == "import":
        sql = text(f"""SELECT hs_code_6d AS hs_code, date, impDlr AS value
                       FROM {TABLE_IMPORT_MONTHLY}
                       WHERE forecast_flag = 0 AND impDlr IS NOT NULL""")
    elif direction == "export":
        with engine.connect() as conn:
            vcol = _export_version_col(conn)
        join = f"""
            JOIN (SELECT hs_code, date_month_end, MAX({vcol}) AS mx
                  FROM {TABLE_EXPORT_MONTHLY}
                  WHERE is_forecast = 0 AND expDlr IS NOT NULL
                  GROUP BY hs_code, date_month_end) l
              ON t.hs_code = l.hs_code AND t.date_month_end = l.date_month_end
             AND t.{vcol} = l.mx """ if vcol else ""
        sql = text(f"""SELECT t.hs_code AS hs_code, t.date_month_end AS date, t.expDlr AS value
                       FROM {TABLE_EXPORT_MONTHLY} t {join}
                       WHERE t.is_forecast = 0 AND t.expDlr IS NOT NULL""")
    else:
        raise ValueError("direction must be 'import' or 'export'")

    with engine.connect() as conn:
        rows = conn.execute(sql).fetchall()
    df = pd.DataFrame(rows, columns=["hs_code", "date", "value"])
    df["date"] = pd.to_datetime(df["date"]).apply(_month_end_calendar)
    df["hs_code"] = df["hs_code"].astype(str).str.zfill(6)

    n_dup = df.duplicated(subset=["hs_code", "date"]).sum()
    if n_dup:
        df = df.drop_duplicates(subset=["hs_code", "date"], keep="last")
    if verbose:
        print(f"[{direction}] {len(df):,}행 / HS {df['hs_code'].nunique():,}개"
              + (f" / 중복 {n_dup:,}행 제거" if n_dup else ""))
    return df


# ---------- 통계 유틸 ----------

def eff_n(n, rho1, rho2):
    """자기상관을 반영한 유효 표본 수 (Bartlett 근사)."""
    prod = np.clip(rho1 * rho2, -0.99, 0.99)
    return np.maximum(n * (1 - prod) / (1 + prod), 4.0)


def corr_rows(a, b):
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    return (a * b).sum(1) / np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1))


def sim_ar1(n, rho, size, rng):
    rho = float(np.clip(rho, -0.98, 0.98))
    e = rng.standard_normal((size, n))
    x = np.empty_like(e)
    x[:, 0] = e[:, 0] / np.sqrt(1 - rho ** 2)
    for t in range(1, n):
        x[:, t] = rho * x[:, t - 1] + e[:, t]
    return x


def null_corr_dist(n, rho1, rho2, sims, rng):
    """지속성을 가진 두 독립 계열의 상관계수 귀무분포."""
    return corr_rows(sim_ar1(n, rho1, sims, rng), sim_ar1(n, rho2, sims, rng))


def newey_west_t(f, lag=2):
    """평균이 0인지 검정 (자기상관 보정). f는 1차원 배열."""
    f = np.asarray(f, float)
    f = f[~np.isnan(f)]
    n = len(f)
    if n < 5:
        return np.nan, np.nan
    mu = f.mean()
    d = f - mu
    v = (d ** 2).mean()
    for L in range(1, min(lag, n - 1) + 1):
        v += 2 * (1 - L / (lag + 1)) * (d[L:] * d[:-L]).mean()
    if v <= 0:
        return np.nan, np.nan
    t = mu / np.sqrt(v / n)
    p = sps.norm.sf(t) if HAS_SCIPY else np.nan
    return t, p

print("헬퍼 로드 완료")


## §1. 데이터 무결성

분석 이전에 데이터 자체가 온전한지 확인합니다. 여기서 걸리면 이후 §는 전부 무의미합니다.

In [ ]:

print("=" * 74)
print("§1-1. 회계 캘린더 스냅 (52/53주 기업)")
print("=" * 74)
DIAG_TICKERS = ("AAPL", "AMAT", "MU", "DE")

chk = pd.read_sql(text(f"""
    SELECT ticker, period, date, value FROM {TABLE_REVENUE}
    WHERE ticker IN ({','.join(repr(t) for t in DIAG_TICKERS)})
      AND item = 'revenue' AND period IN ('Q1','Q2','Q3','Q4')
      AND value IS NOT NULL AND value > 0
"""), engine)
chk["date"] = pd.to_datetime(chk["date"])

print("중복 점검 (count == nunique 여야 정상):")
print(chk.groupby("ticker")["date"].agg(["count", "nunique"]).to_string(), "\n")

_snap_ok = True
for tk, g in chk.groupby("ticker"):
    raw = sorted(g["date"].unique())
    line = [f"{tk:5s} 원본 {len(raw):3d}분기"]
    for label, fn in [("calendar", _month_end_calendar), ("fiscal", _month_end_fiscal)]:
        s = sorted({fn(d) for d in raw})
        run = get_trailing_consecutive_run(s, QUARTER_TOL_DAYS)
        line.append(f"{label}={len(run):3d}")
        if label == "fiscal" and len(run) < len(raw) * 0.9:
            _snap_ok = False
    # 월초로 넘어간 분기말 (버그를 유발하는 날짜)
    spill = [str(pd.Timestamp(d).date()) for d in raw if pd.Timestamp(d).day <= 3]
    print("  ".join(line) + (f"   월초 분기말: {spill[:4]}" if spill else ""))
print(f"\n판정: {'통과' if _snap_ok else '실패 — _month_end_fiscal 적용 여부 확인'}")


In [ ]:

print("=" * 74)
print("§1-2. 수출 테이블 self-join 영향도")
print("=" * 74)
if DIRECTION == "export":
    with engine.connect() as conn:
        vcol = _export_version_col(conn)
    print(f"버전 컬럼: {vcol}")
    if vcol:
        q = text(f"""
        SELECT
          (SELECT COUNT(*) FROM {TABLE_EXPORT_MONTHLY}
             WHERE is_forecast=0 AND expDlr IS NOT NULL) AS raw_rows,
          (SELECT COUNT(*) FROM {TABLE_EXPORT_MONTHLY} t
             JOIN (SELECT hs_code, MAX({vcol}) mx FROM {TABLE_EXPORT_MONTHLY}
                   WHERE is_forecast=0 AND expDlr IS NOT NULL GROUP BY hs_code) l
               ON t.hs_code=l.hs_code AND t.{vcol}=l.mx
             WHERE t.is_forecast=0 AND t.expDlr IS NOT NULL) AS old_join,
          (SELECT COUNT(*) FROM {TABLE_EXPORT_MONTHLY} t
             JOIN (SELECT hs_code, date_month_end, MAX({vcol}) mx FROM {TABLE_EXPORT_MONTHLY}
                   WHERE is_forecast=0 AND expDlr IS NOT NULL
                   GROUP BY hs_code, date_month_end) l
               ON t.hs_code=l.hs_code AND t.date_month_end=l.date_month_end AND t.{vcol}=l.mx
             WHERE t.is_forecast=0 AND t.expDlr IS NOT NULL) AS new_join
        """)
        s = pd.read_sql(q, engine).iloc[0]
        print(f"  원본            {s['raw_rows']:>12,}")
        print(f"  구버전 join     {s['old_join']:>12,}  ({s['old_join']/s['raw_rows']*100:5.1f}%)")
        print(f"  수정 join       {s['new_join']:>12,}  ({s['new_join']/s['raw_rows']*100:5.1f}%)")
        verdict = "통과" if s['old_join'] >= s['new_join'] * 0.95 else "구버전 join이 데이터를 대량 유실 — 수정본 사용 필수"
        print(f"\n판정: {verdict}")
else:
    print("  (수입 모드 — self-join 없음, 건너뜀)")

print()
print("=" * 74)
print("§1-3. 시점 정합성 (개정 이력)")
print("=" * 74)
print("백테스트는 '당시 알 수 있었던 값'을 써야 하는데, 테이블에 최종 개정치만 있으면")
print("look-ahead bias가 생깁니다. 같은 (hs_code, 월)에 값이 여러 버전 있는지 확인합니다.\n")
if DIRECTION == "export":
    with engine.connect() as conn:
        vcol = _export_version_col(conn)
    if vcol:
        rev_hist = pd.read_sql(text(f"""
            SELECT hs_code, date_month_end,
                   COUNT(DISTINCT expDlr) AS n_versions,
                   MIN(expDlr) AS v_min, MAX(expDlr) AS v_max
            FROM {TABLE_EXPORT_MONTHLY}
            WHERE is_forecast=0 AND expDlr IS NOT NULL
            GROUP BY hs_code, date_month_end
            HAVING n_versions > 1
            ORDER BY date_month_end DESC LIMIT 200
        """), engine)
        if len(rev_hist):
            rev_hist["diff_pct"] = (rev_hist.v_max/rev_hist.v_min - 1)*100
            print(f"  복수 버전 존재: {len(rev_hist):,}건 (상위 200개 표본)")
            print(f"  개정폭 중앙값 {rev_hist.diff_pct.median():.2f}% / 최대 {rev_hist.diff_pct.max():.2f}%")
            print("\n  → 초판 값으로 백테스트를 다시 돌리는 것을 권장합니다.")
        else:
            print("  복수 버전 없음 — 개정 이력이 보관되지 않았습니다.")
            print("  → §6 백테스트 결과는 개정치를 쓴 것이므로 낙관 편향이 있을 수 있습니다.")


## §2. 패널 구축

매출 YoY(3그룹)와 무역 YoY 패널을 만듭니다. 이후 모든 §가 이 두 객체를 씁니다.

In [ ]:

# ---------- 매출 ----------
with engine.connect() as conn:
    _cols = {r[0] for r in conn.execute(text(f"SHOW COLUMNS FROM {TABLE_REVENUE}")).fetchall()}
_ob = "ticker, date, id" if "id" in _cols else "ticker, date"

rows = pd.read_sql(text(f"""
    SELECT ticker, date, value AS revenue FROM {TABLE_REVENUE}
    WHERE item='revenue' AND period IN ('Q1','Q2','Q3','Q4')
      AND value IS NOT NULL AND value > 0
    ORDER BY {_ob}
"""), engine)
revenue_raw = rows.copy()
revenue_raw["date"] = pd.to_datetime(revenue_raw["date"]).apply(_month_end_fiscal)
revenue_raw = revenue_raw.drop_duplicates(subset=["ticker", "date"], keep="last")
print(f"[매출] 티커 {revenue_raw.ticker.nunique():,} / 레코드 {len(revenue_raw):,}")

kept = []
for tk, g in revenue_raw.groupby("ticker"):
    run = get_trailing_consecutive_run(g["date"].tolist(), QUARTER_TOL_DAYS)
    if len(run) >= MIN_QUARTERS:
        kept.append(g[g["date"].isin(run)])
if not kept:
    raise RuntimeError(f"연속 {MIN_QUARTERS}분기 티커 없음 — DB 분기 수를 확인하세요.")
revenue_filtered = pd.concat(kept, ignore_index=True)
revenue_filtered["cycle_group"] = revenue_filtered["date"].dt.month % 3
print(f"[매출] 연속 {MIN_QUARTERS}분기 통과: {revenue_filtered.ticker.nunique():,}개")

# cycle_group 단일화
_st = revenue_filtered.groupby("ticker")["cycle_group"].nunique()
if (_st > 1).any():
    dom = revenue_filtered.groupby("ticker")["cycle_group"].agg(lambda s: s.mode().iat[0])
    n0 = len(revenue_filtered)
    revenue_filtered = revenue_filtered[
        revenue_filtered.cycle_group == revenue_filtered.ticker.map(dom)].copy()
    print(f"[매출] cycle_group 다중 티커 {int((_st>1).sum()):,}개 통일 ({n0-len(revenue_filtered):,}행 제외)")

GROUP_LABELS = {0: "표준그룹(3,6,9,12월분기말)", 1: "그룹B(4,7,10,1월분기말)", 2: "그룹C(5,8,11,2월분기말)"}
revenue_yoy_by_group = {}
for gid, label in GROUP_LABELS.items():
    piv = (revenue_filtered[revenue_filtered.cycle_group == gid]
           .pivot_table(index="date", columns="ticker", values="revenue", aggfunc="last"))
    parts = []
    for tk in piv.columns:
        y = compute_revenue_yoy(piv[tk].dropna())
        if not y.empty:
            t = y.reset_index(); t.columns = ["date", "revenue_yoy"]; t["ticker"] = tk
            parts.append(t)
    revenue_yoy_by_group[label] = (pd.concat(parts, ignore_index=True)[["ticker","date","revenue_yoy"]]
                                   if parts else pd.DataFrame(columns=["ticker","date","revenue_yoy"]))
    print(f"  - {label}: 티커 {revenue_yoy_by_group[label].ticker.nunique():,}")

# ---------- 무역 ----------
trade_long = load_trade_monthly_long(engine, DIRECTION)
kept = []
for hs, g in trade_long.groupby("hs_code"):
    run = get_trailing_consecutive_run(g["date"].tolist(), MONTH_TOL_DAYS)
    if len(run) >= MIN_MONTHS:
        kept.append(g[g["date"].isin(run)])
if not kept:
    raise RuntimeError(f"연속 {MIN_MONTHS}개월 HS코드 없음 — §1-2 self-join 결과를 확인하세요.")
trade_filtered = pd.concat(kept, ignore_index=True)
monthly_panel = trade_filtered.pivot_table(index="date", columns="hs_code", values="value", aggfunc="sum")
trade_yoy_panel = monthly_panel.rolling(3, min_periods=3).sum().pct_change(12) * 100.0
print(f"[{DIRECTION}] 통과 HS {trade_yoy_panel.shape[1]:,}개 / 패널 {trade_yoy_panel.shape}")
print(f"[{DIRECTION}] 기간 {trade_yoy_panel.index.min():%Y-%m} ~ {trade_yoy_panel.index.max():%Y-%m}")

# ---------- 조회 헬퍼 ----------
TICKER_GROUP = {}
for label, df in revenue_yoy_by_group.items():
    for tk in df.ticker.unique():
        TICKER_GROUP[tk] = label

def get_pair(ticker, hs, shift=0):
    """(rev_yoy, trade_yoy, group) 정렬된 시리즈 반환. 없으면 None."""
    g = TICKER_GROUP.get(ticker)
    hs = str(hs).zfill(6)
    if g is None or hs not in trade_yoy_panel.columns:
        return None
    rev = (revenue_yoy_by_group[g].query("ticker == @ticker")
           .set_index("date")["revenue_yoy"].sort_index())
    tr = trade_yoy_panel[hs].shift(shift).reindex(rev.index)
    m = rev.notna() & tr.notna()
    if m.sum() < MIN_CORR_PERIODS:
        return None
    return rev[m], tr[m], g

print("\n패널 구축 완료. get_pair(ticker, hs, shift) 로 조회하세요.")


## §3. 잡음 바닥 — 관측 상관계수가 우연을 넘는가

**가장 중요한 단계입니다.** YoY 계열은 자기상관이 높아, 아무 관계 없는 두 계열도
|r| 0.4~0.5를 흔히 만듭니다. 실측 자기상관으로 귀무분포를 만들어
"이 티커의 상관계수 분포가 잡음보다 위인가"를 봅니다.

**통과 기준**: 관측 초과 개수 / 귀무 기대 개수 ≥ 2.0

In [ ]:

def noise_floor_report(ticker, thresholds=(0.4, 0.5, 0.6, 0.7), sims=N_SIM, verbose=True):
    """한 티커의 전체 HS 상관계수 분포를 귀무분포와 비교."""
    g = TICKER_GROUP.get(ticker)
    if g is None:
        print(f"{ticker}: 매출 패널에 없음"); return None
    rev = (revenue_yoy_by_group[g].query("ticker == @ticker")
           .set_index("date")["revenue_yoy"].sort_index())
    aligned = trade_yoy_panel.reindex(rev.index)

    r_obs = aligned.corrwith(rev)
    n_obs = (aligned.notna() & rev.notna().values[:, None]).sum(axis=0)
    keep = n_obs[n_obs >= MIN_CORR_PERIODS].index
    r_obs = r_obs.loc[keep].dropna()
    n_used = int(n_obs.loc[keep].median())

    rho_rev = rev.autocorr(1)
    rho_tr = aligned[keep].apply(lambda s: s.autocorr(1)).median()
    null = np.abs(null_corr_dist(n_used, rho_rev, rho_tr, sims, RNG))

    if verbose:
        print(f"\n{'='*74}\n{ticker}  (그룹: {g})")
        print(f"  HS {len(r_obs):,}개 / 중앙 n={n_used} / "
              f"AR(1) 매출={rho_rev:.3f}, 무역={rho_tr:.3f} / 유효 n≈{eff_n(n_used,rho_rev,rho_tr):.1f}")
        print(f"  {'기준':>6} {'관측':>7} {'귀무기대':>9} {'배율':>7}")
        print(f"  {'-'*34}")
    rows = []
    for th in thresholds:
        obs = int((r_obs.abs() >= th).sum())
        exp = float((null >= th).mean() * len(r_obs))
        ratio = obs / exp if exp > 0.5 else np.nan
        rows.append(dict(threshold=th, observed=obs, expected=exp, ratio=ratio))
        if verbose:
            r_s = f"{ratio:7.2f}" if np.isfinite(ratio) else "      -"
            print(f"  {th:6.1f} {obs:7d} {exp:9.1f} {r_s}")
    out = pd.DataFrame(rows)
    if verbose:
        core = out.query("threshold in [0.5, 0.6]")["ratio"].dropna()
        v = core.mean() if len(core) else np.nan
        print(f"  판정: 평균 배율 {v:.2f} → "
              f"{'통과 (잡음 초과)' if v >= 2 else '실패 (잡음과 구분 안 됨)'}")
    return out


# 검증군: 채널이 있어야 하는 종목 vs 없어야 하는 종목
for tk in ["MU", "AMAT", "DE", "APH", "LLY"]:
    noise_floor_report(tk)


## §4. 사전 지정 가설 — 산업 논리로 미리 정한 쌍이 상위에 오는가

스캔 결과를 보고 사후에 해석하면 반드시 그럴듯한 이야기가 만들어집니다.
**데이터를 보기 전에** 산업 논리로 쌍을 정하고, 그것이 1,500여 개 티커 중
몇 위인지 봅니다. 이 검정만이 다중비교로부터 자유롭습니다.

**통과 기준**: 가설의 절반 이상이 상위 1% 이내, 이항검정 p < 0.01

In [ ]:

# ★ 데이터를 보기 전에 작성할 것 ★  ticker -> (hs_code, 근거)
HYPOTHESES = {
    "MU":   ("854232", "메모리 집적회로"),
    "AMAT": ("848620", "반도체 제조장비"),
    "LRCX": ("848620", "반도체 제조장비"),
    "DE":   ("843351", "콤바인 수확기"),
    "CAT":  ("842952", "굴착기"),
    "BA":   ("880240", "항공기 15톤 초과"),
    "NUE":  ("720719", "철강 반제품"),
    "MOS":  ("310420", "칼륨비료"),
    "FCX":  ("740311", "정제동 캐소드"),
    # 대조군 (채널이 없어야 정상)
    "LLY":  ("300439", "[대조군] 호르몬 의약품 — 아일랜드 생산이라 미약 예상"),
}

def hypothesis_rank(ticker, hs, min_n=40):
    hs = str(hs).zfill(6)
    if hs not in trade_yoy_panel.columns:
        return dict(status="HS코드 패널에 없음")
    if ticker not in TICKER_GROUP:
        return dict(status="티커 매출 패널에 없음")

    rows = []
    for label, df in revenue_yoy_by_group.items():
        piv = df.pivot_table(index="date", columns="ticker", values="revenue_yoy", aggfunc="last")
        for tk in piv.columns:
            s = piv[tk].dropna()
            tr = trade_yoy_panel[hs].reindex(s.index)
            m = s.notna() & tr.notna()
            if m.sum() >= min_n:
                rows.append((tk, np.corrcoef(s[m], tr[m])[0, 1], int(m.sum())))
    if not rows:
        return dict(status="비교 대상 없음")
    tbl = (pd.DataFrame(rows, columns=["ticker", "r", "n"])
           .sort_values("r", ascending=False).reset_index(drop=True))
    hit = tbl.index[tbl.ticker == ticker]
    if len(hit) == 0:
        return dict(status=f"관측 수 {min_n} 미만")
    i = int(hit[0])
    return dict(status="ok", rank=i + 1, total=len(tbl),
                pct=(i + 1) / len(tbl) * 100, r=tbl.loc[i, "r"], n=int(tbl.loc[i, "n"]))


print(f"{'티커':<6}{'HS':<8}{'순위':>12} {'상위%':>7} {'r':>7} {'n':>5}  근거")
print("-" * 88)
hyp_results = {}
for tk, (hs, why) in HYPOTHESES.items():
    res = hypothesis_rank(tk, hs)
    hyp_results[tk] = res
    if res["status"] != "ok":
        print(f"{tk:<6}{hs:<8}{res['status']:>12}")
    else:
        print(f"{tk:<6}{hs:<8}{res['rank']:>6d}/{res['total']:<5d}"
              f"{res['pct']:6.2f}% {res['r']:+7.3f} {res['n']:5d}  {why}")

# 이항검정 (대조군 제외)
ok = {k: v for k, v in hyp_results.items()
      if v["status"] == "ok" and not HYPOTHESES[k][1].startswith("[대조군]")}
if ok and HAS_SCIPY:
    N_TOTAL = int(np.median([v["total"] for v in ok.values()]))
    k_top1 = sum(v["pct"] <= 1.0 for v in ok.values())
    p_each = 0.01
    p_val = sps.binomtest(k_top1, len(ok), p_each, alternative="greater").pvalue
    print(f"\n사전 지정 {len(ok)}개 중 상위 1% 진입: {k_top1}개")
    print(f"이항검정 p = {p_val:.3e}  → "
          f"{'통과' if (p_val < 0.01 and k_top1 >= len(ok)/2) else '미달'}")


## §5. 선행/동행 구조

무역 데이터가 매출을 **선행**하는지, 동행할 뿐인지 확인합니다.
shift가 양수면 그만큼 과거의 무역 데이터를 쓴다는 뜻입니다(=무역이 선행).

동시에, 관측된 최고점 위치가 **식별 가능한 것인지**도 시뮬레이션으로 검증합니다.
지속성이 높은 계열끼리는 최고점이 우연히 옆으로 밀릴 수 있기 때문입니다.

In [ ]:

def lead_lag_profile(ticker, hs, max_shift=6, plot=True):
    rows = []
    for k in range(-max_shift, max_shift + 1):
        p = get_pair(ticker, hs, shift=k)
        if p is None:
            continue
        rev, tr, _ = p
        rows.append((k, float(np.corrcoef(rev, tr)[0, 1]), len(rev)))
    if not rows:
        return None
    df = pd.DataFrame(rows, columns=["shift", "r", "n"])
    best = df.loc[df.r.idxmax()]
    r0 = df.query("shift == 0")["r"]
    r0 = float(r0.iloc[0]) if len(r0) else np.nan
    if plot:
        bars = "  ".join(f"{int(s):+d}:{r:+.2f}" for s, r in zip(df["shift"], df["r"]))
        print(f"{ticker:5s} x {hs}  최고점 shift={int(best['shift']):+d} "
              f"(r={best['r']:+.3f}, shift0 r={r0:+.3f}, 차이 {best['r']-r0:+.3f})")
        print(f"       {bars}")
    return df


def peak_identifiability(rho=0.85, n_q=49, sims=1500, max_shift=6):
    """진짜 지연이 0일 때 추정 최고점이 어디로 흩어지는지 + 선택편향 크기."""
    nm = n_q * 3 + 40
    peaks, gap = [], []
    for _ in range(sims):
        z = sim_ar1(nm, rho, 1, RNG)[0]
        tr = z + 0.7 * sim_ar1(nm, rho, 1, RNG)[0]
        rv = z + 0.7 * sim_ar1(nm, rho, 1, RNG)[0]
        qi = np.arange(24, nm - max_shift - 1, 3)
        y = rv[qi]
        prof = {}
        for k in range(-max_shift, max_shift + 1):
            x = tr[qi - k]
            a, b = y - y.mean(), x - x.mean()
            prof[k] = (a * b).sum() / np.sqrt((a ** 2).sum() * (b ** 2).sum())
        bk = max(prof, key=prof.get)
        peaks.append(bk); gap.append(prof[bk] - prof[0])
    peaks = np.array(peaks); gap = np.array(gap)
    print(f"\n[식별력] 진짜 지연=0, 지속성={rho} 일 때 추정 최고점 분포 ({sims:,}회)")
    for k in range(-max_shift, max_shift + 1):
        c = (peaks == k).mean() * 100
        if c >= 0.5:
            print(f"   {k:+d}개월 {c:5.1f}%  {'#' * int(c/2)}{'  <- 진짜' if k==0 else ''}")
    print(f"   최고점≠0 확률 {(peaks!=0).mean()*100:.1f}% / "
          f"선택편향에 의한 r 상승 평균 {gap.mean():+.4f}")
    print("   → 관측된 (최고점 r - shift0 r) 이 이 값보다 훨씬 크면 실제 선행입니다.")


print("=" * 74)
for tk, (hs, _) in HYPOTHESES.items():
    if get_pair(tk, hs) is not None:
        lead_lag_profile(tk, hs)
        print()

peak_identifiability()


## §6. 시점 정합 백테스트 — AR(1) 위에 증분 정보가 있는가

**여기가 실용성의 관문입니다.**

매출 YoY는 지속성이 높아 직전 분기 값(AR(1))만으로도 잘 맞습니다.
무역 데이터의 가치는 AR(1)을 **얼마나 개선하는가**로만 측정해야 합니다.
중첩 모델 비교이므로 일반 DM 검정 대신 **Clark-West**를 씁니다.

**통과 기준**: 결합모델이 AR(1) 대비 RMSE 15% 이상 개선 **그리고** Clark-West p < 0.05

In [ ]:

def pit_backtest(ticker, hs, shift=1, train_min=16):
    """확장 윈도우. 각 시점에서 과거 데이터만 사용해 예측."""
    p = get_pair(ticker, hs, shift=shift)
    if p is None:
        return None
    rev, tr, g = p
    if len(rev) < train_min + 6:
        return None

    rows = []
    for i in range(train_min, len(rev)):
        y = rev.iloc[1:i].values          # 타깃
        L = rev.iloc[0:i-1].values        # 직전 분기 YoY
        X = tr.iloc[1:i].values           # 무역 YoY
        y_last, x_now = rev.iloc[i-1], tr.iloc[i]
        if len(y) < 8:
            continue
        b1, a1 = np.polyfit(X, y, 1)
        b2, a2 = np.polyfit(L, y, 1)
        A = np.column_stack([np.ones(len(y)), L, X])
        c = np.linalg.lstsq(A, y, rcond=None)[0]
        rows.append(dict(date=rev.index[i], actual=rev.iloc[i],
                         trade=a1 + b1 * x_now,
                         ar1=a2 + b2 * y_last,
                         both=c[0] + c[1] * y_last + c[2] * x_now,
                         mean=y.mean()))
    if len(rows) < 10:
        return None
    df = pd.DataFrame(rows)
    rmse = {k: float(np.sqrt(((df[k] - df.actual) ** 2).mean()))
            for k in ["trade", "ar1", "both", "mean"]}
    return df, rmse


def clark_west(df, restricted="ar1", unrestricted="both", lag=2):
    y = df["actual"].values
    f1, f2 = df[restricted].values, df[unrestricted].values
    f = (y - f1) ** 2 - ((y - f2) ** 2 - (f1 - f2) ** 2)
    return newey_west_t(f, lag)


print(f"{'티커':<6}{'무역단독':>9}{'AR(1)':>9}{'결합':>9}{'평균':>9} | "
      f"{'결합vsAR1':>10} {'CW t':>7} {'CW p':>8}  판정")
print("-" * 84)
bt_results = {}
for tk, (hs, _) in HYPOTHESES.items():
    out = pit_backtest(tk, hs)
    if out is None:
        print(f"{tk:<6}{'데이터 부족':>20}")
        continue
    df, r = out
    gain = (1 - r["both"] / r["ar1"]) * 100
    t, p = clark_west(df)
    ok = (gain >= 15) and (p == p) and (p < 0.05)
    bt_results[tk] = dict(gain=gain, cw_t=t, cw_p=p, rmse=r, n=len(df))
    ps = f"{p:8.4f}" if p == p else "       -"
    ts = f"{t:7.2f}" if t == t else "      -"
    print(f"{tk:<6}{r['trade']:9.2f}{r['ar1']:9.2f}{r['both']:9.2f}{r['mean']:9.2f} | "
          f"{gain:+9.1f}% {ts} {ps}  {'통과' if ok else '미달'}")

print("\n※ AR(1)이 무역단독보다 낮으면, 높은 상관계수는 '두 계열 모두 느리게 움직인다'는")
print("   사실을 반영한 것일 뿐 무역 데이터의 정보력이 아닙니다.")


## §7. 방향성 · 전환점

전체 RMSE는 평범한 분기가 대부분을 차지해 희석됩니다. AR(1)은 원리적으로
**전환점을 예측할 수 없으므로**, 무역 데이터가 전환점에서만 기여한다면
전체 개선폭은 작아도 실용 가치가 있을 수 있습니다.

여기서 보는 것은 YoY의 **부호**가 아니라 **변화 방향(가속/감속)** 입니다.

**통과 기준**: 전환점 구간 개선폭이 전체 구간 개선폭의 2배 이상,
그리고 방향 적중률이 AR(1)보다 높은 종목이 과반

In [ ]:

def directional_test(ticker, hs, shift=1, tp_quantile=0.6):
    out = pit_backtest(ticker, hs, shift)
    if out is None:
        return None
    df, _ = out
    d = df.assign(prev=df["actual"].shift(1)).dropna(subset=["prev"])
    if len(d) < 10:
        return None

    act = np.sign(d["actual"] - d["prev"])
    hit = {k: float((np.sign(d[k] - d["prev"]) == act).mean()) for k in ["ar1", "both"]}

    delta = (d["actual"] - d["prev"]).abs()
    big = delta >= delta.quantile(tp_quantile)
    tp = {k: float(np.sqrt(((d.loc[big, k] - d.loc[big, "actual"]) ** 2).mean()))
          for k in ["ar1", "both"]}
    calm = {k: float(np.sqrt(((d.loc[~big, k] - d.loc[~big, "actual"]) ** 2).mean()))
            for k in ["ar1", "both"]}

    a_ok = np.sign(d["ar1"] - d["prev"]) == act
    b_ok = np.sign(d["both"] - d["prev"]) == act
    b_only, a_only = int((b_ok & ~a_ok).sum()), int((a_ok & ~b_ok).sum())
    p_mc = (sps.binomtest(b_only, b_only + a_only, 0.5, alternative="greater").pvalue
            if HAS_SCIPY and (b_only + a_only) > 0 else np.nan)

    return dict(n=len(d), hit_ar1=hit["ar1"], hit_both=hit["both"],
                tp_gain=(1 - tp["both"] / tp["ar1"]) * 100,
                calm_gain=(1 - calm["both"] / calm["ar1"]) * 100,
                b_only=b_only, a_only=a_only, p_mc=p_mc)


print(f"{'티커':<6}{'n':>4}{'AR1적중':>9}{'결합적중':>9} | "
      f"{'전환점개선':>11}{'평상시개선':>11} | {'McNemar':>14}")
print("-" * 78)
dir_results = {}
for tk, (hs, _) in HYPOTHESES.items():
    r = directional_test(tk, hs)
    if r is None:
        continue
    dir_results[tk] = r
    ps = f"p={r['p_mc']:.3f}" if r["p_mc"] == r["p_mc"] else "p=-"
    print(f"{tk:<6}{r['n']:4d}{r['hit_ar1']:8.1%}{r['hit_both']:9.1%} | "
          f"{r['tp_gain']:+10.1f}%{r['calm_gain']:+10.1f}% | "
          f"{r['b_only']:2d}:{r['a_only']:<2d} {ps:>9}")

if dir_results:
    n_better = sum(v["hit_both"] > v["hit_ar1"] for v in dir_results.values())
    tp_mean = np.mean([v["tp_gain"] for v in dir_results.values()])
    calm_mean = np.mean([v["calm_gain"] for v in dir_results.values()])
    print(f"\n적중률 개선 종목 {n_better}/{len(dir_results)} | "
          f"평균 전환점 개선 {tp_mean:+.1f}% vs 평상시 {calm_mean:+.1f}%")
    print(f"판정: {'통과 — 전환점에서 더 크게 기여' if tp_mean > max(calm_mean*2, 10) and n_better*2 >= len(dir_results) else '미달 — 전환점 특화 효과 없음'}")

print("\n※ n≈32에서 방향 적중률이 유의하려면 69%(22/32) 이상 필요합니다.")
print("   개별 유의성보다 '여러 종목에서 일관된 방향'을 보세요.")


## §8. 괴리 탐지

평상시 동행할 때는 새 정보가 없습니다. 무역과 매출이 **벌어지는** 드문 구간이
오히려 신호일 수 있습니다 — 재고 조정, 지역별 수요 분화, 생산지 이전 등.

예측 정확도 논쟁과 무관하게 쓸 수 있는 용도입니다.

In [ ]:

def divergence_scan(ticker, hs, shift=1, z_thresh=1.5):
    out = pit_backtest(ticker, hs, shift)
    if out is None:
        return None
    df, _ = out
    gap = df["actual"] - df["trade"]
    z = (gap - gap.mean()) / gap.std(ddof=1)
    df = df.assign(gap=gap, gap_z=z)
    return df.loc[df.gap_z.abs() >= z_thresh,
                  ["date", "actual", "trade", "gap", "gap_z"]].sort_values("date")


for tk, (hs, _) in HYPOTHESES.items():
    d = divergence_scan(tk, hs)
    if d is None or d.empty:
        continue
    print(f"\n{'='*74}\n{tk} x {hs}  괴리 구간 {len(d)}개 (|z| >= 1.5)")
    print(d.assign(date=d.date.dt.strftime("%Y-%m")).to_string(index=False,
          float_format=lambda v: f"{v:8.2f}"))


## §9. 종합 스코어카드

각 단계의 판정을 한 표로 모읍니다. 사후에 기준을 조정하지 마세요.

In [ ]:

rows = []
for tk, (hs, why) in HYPOTHESES.items():
    h = hyp_results.get(tk, {})
    b = bt_results.get(tk, {})
    d = dir_results.get(tk, {})
    ll = lead_lag_profile(tk, hs, plot=False)
    peak = int(ll.loc[ll.r.idxmax(), "shift"]) if ll is not None and len(ll) else None
    rows.append(dict(
        ticker=tk, hs=hs,
        rank=f"{h.get('rank','-')}/{h.get('total','-')}" if h.get("status") == "ok" else h.get("status", "-"),
        top_pct=round(h["pct"], 2) if h.get("status") == "ok" else np.nan,
        r=round(h["r"], 3) if h.get("status") == "ok" else np.nan,
        peak_shift=peak,
        ar1_rmse=round(b["rmse"]["ar1"], 2) if b else np.nan,
        both_rmse=round(b["rmse"]["both"], 2) if b else np.nan,
        gain_pct=round(b["gain"], 1) if b else np.nan,
        cw_p=round(b["cw_p"], 4) if b and b["cw_p"] == b["cw_p"] else np.nan,
        tp_gain=round(d["tp_gain"], 1) if d else np.nan,
        control="Y" if why.startswith("[대조군]") else "",
    ))
score = pd.DataFrame(rows)

def verdict(r):
    if pd.isna(r.gain_pct):
        return "데이터부족"
    if r.gain_pct >= 15 and pd.notna(r.cw_p) and r.cw_p < 0.05:
        return "PASS"
    if pd.notna(r.tp_gain) and r.tp_gain >= 25:
        return "전환점만"
    if pd.notna(r.top_pct) and r.top_pct <= 1:
        return "매핑만"
    return "FAIL"

score["verdict"] = score.apply(verdict, axis=1)
print(score.to_string(index=False))

print("\n" + "=" * 74)
print("판정 의미")
print("=" * 74)
print("  PASS      : 예측 도구로 사용 가능. 다음은 애널리스트 컨센서스 대비 검정.")
print("  전환점만  : 상시 예측은 불가. 전환점 감지 보조지표로만 사용.")
print("  매핑만    : 예측력은 없으나 기업-무역흐름 노출도 식별에는 유효.")
print("              (관세 시나리오, 공급망 리스크 스크리닝 용도)")
print("  FAIL      : 해당 쌍은 사용하지 말 것.")
print()
print("대조군(LLY)이 FAIL 이고 사전 지정군에 PASS/전환점만 이 섞여 있으면")
print("파이프라인이 정상 작동하는 것입니다. 대조군이 PASS 면 검정 설계에 문제가 있습니다.")

out_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_trade_revenue_corr"
os.makedirs(out_dir, exist_ok=True)
fp = os.path.join(out_dir, f"trade_validation_{DIRECTION}_{pd.Timestamp.today():%Y%m%d}.csv")
score.to_csv(fp, index=False, encoding="utf-8-sig")
print(f"\n[저장] {fp}")
